# Price data preprocessing
Bu notebook BTC/USD bir-dakikalık verisini saatlik özelliklere çevirir ve teknik indikatörler ekler.

In [7]:
import os
import pandas as pd
import numpy as np
import ta

root = os.path.expanduser("~/Desktop/yzproje")

raw_file = os.path.join(root, "data", "raw", "bitcoin", "btcusd_1-min_data.csv")
out_dir = os.path.join(root, "data", "processed")
out_file = os.path.join(out_dir, "bitcoin_hourly_features.csv")

os.makedirs(out_dir, exist_ok=True)

print("Reading:", raw_file)

df = pd.read_csv(raw_file)
print("Loaded rows:", len(df))

print("\nFirst 5 rows:")
print(df.head())

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing counts:")
print(df.isna().sum())

df["Timestamp"] = pd.to_datetime(df["Timestamp"], unit="s")
df = df.set_index("Timestamp")
df = df.sort_index()

df_hour = df.resample("h").agg({
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum"
})

df_hour = df_hour.ffill()
df_hour = df_hour.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
})

df_hour["rsi"] = ta.momentum.RSIIndicator(df_hour["close"]).rsi()

macd = ta.trend.MACD(df_hour["close"])
df_hour["macd"] = macd.macd()
df_hour["macd_signal"] = macd.macd_signal()

bb = ta.volatility.BollingerBands(df_hour["close"])
df_hour["bollinger_h"] = bb.bollinger_hband()
df_hour["bollinger_l"] = bb.bollinger_lband()

df_hour["return"] = df_hour["close"].pct_change()
df_hour["future_volatility"] = df_hour["return"].rolling(window=24).std().shift(-24)

df_clean = df_hour.dropna()

df_clean.to_csv(out_file)

print("\nSaved to:", out_file)

print("\nFirst 5 processed rows:")
print(df_clean.head())

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nShape:")
print(df_clean.shape)

print("\nFile exists:", os.path.exists(out_file))

Reading: /Users/hayrunnisabusraerdem/Desktop/yzproje/data/raw/bitcoin/btcusd_1-min_data.csv
Loaded rows: 7566897

First 5 rows:
      Timestamp  Open  High   Low  Close  Volume
0  1.325412e+09  4.58  4.58  4.58   4.58     0.0
1  1.325412e+09  4.58  4.58  4.58   4.58     0.0
2  1.325412e+09  4.58  4.58  4.58   4.58     0.0
3  1.325412e+09  4.58  4.58  4.58   4.58     0.0
4  1.325412e+09  4.58  4.58  4.58   4.58     0.0

Columns:
['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']

Missing counts:
Timestamp    0
Open         0
High         0
Low          0
Close        0
Volume       0
dtype: int64
Loaded rows: 7566897

First 5 rows:
      Timestamp  Open  High   Low  Close  Volume
0  1.325412e+09  4.58  4.58  4.58   4.58     0.0
1  1.325412e+09  4.58  4.58  4.58   4.58     0.0
2  1.325412e+09  4.58  4.58  4.58   4.58     0.0
3  1.325412e+09  4.58  4.58  4.58   4.58     0.0
4  1.325412e+09  4.58  4.58  4.58   4.58     0.0

Columns:
['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volum